In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')


In [4]:
# Simulated data with labels - for XGBoost (supervised)
features = pd.read_csv("../data/features_with_rules.csv")

In [5]:


# Real YouTube data - for Isolation Forest (unsupervised)
yt_df = pd.read_csv("/Users/shamilikolli/Downloads/videoStatististicsALL_from21_anonymized.csv", sep=";")

In [16]:
yt_df.tail()

,Unnamed: 0,video_id,created_at,comments,dislikes,likes,views,like_view_ratio,comment_view_ratio,dislike_view_ratio
45593750,45593750,370132,2022-04-14 15:01:02,6.0,0.0,37.0,1004,0.036816,0.005970,0.0
45593751,45593751,370132,2022-04-14 16:01:05,6.0,0.0,38.0,1005,0.037773,0.005964,0.0
45593752,45593752,370132,2022-04-14 17:01:04,6.0,0.0,38.0,1011,0.037549,0.005929,0.0
45593753,45593753,370132,2022-04-14 18:01:03,6.0,0.0,38.0,1018,0.037291,0.005888,0.0
45593754,45593754,370132,2022-04-14 19:01:04,6.0,0.0,38.0,1018,0.037291,0.005888,0.0


In [17]:
yt_df["created_at"] = pd.to_datetime(yt_df["created_at"])

print(f"Full dataset rows    : {len(yt_df):,}")
print(f"Unique videos        : {yt_df['video_id'].nunique():,}")
print(f"Date range           : {yt_df['created_at'].min()} → {yt_df['created_at'].max()}")
print(f"Avg rows per video   : {len(yt_df)/yt_df['video_id'].nunique():.1f}")
print(f"\nMemory usage: {yt_df.memory_usage(deep=True).sum()/1024/1024:.1f} MB")

Full dataset rows    : 45,593,755
Unique videos        : 270,133
Date range           : 2021-01-01 00:00:28 → 2022-05-10 11:01:11
Avg rows per video   : 168.8

Memory usage: 3478.5 MB


In [9]:
yt_df['video_id'].nunique()

270133

In [7]:
yt_df["like_view_ratio"] = yt_df["likes"] / (yt_df["views"] + 1)
yt_df["comment_view_ratio"] = yt_df["comments"] / (yt_df["views"] + 1)
yt_df["dislike_view_ratio"] = yt_df["dislikes"] / (yt_df["views"] + 1)

print(f"Simulated features : {features.shape}")
print(f"Real YouTube data  : {yt_df.shape}")
print(f"\nYouTube sample:")
print(yt_df[["video_id","views","likes","like_view_ratio"]].head(5))

Simulated features : (1000, 20)
Real YouTube data  : (45593755, 10)

YouTube sample:
   video_id  views  likes  like_view_ratio
0    100000    162   16.0         0.098160
1    100000    266   36.0         0.134831
2    100000   1474   49.0         0.033220
3    100000   3746   56.0         0.014945
4    100000   5625   62.0         0.011020


In [11]:
yt_df["created_at"] = pd.to_datetime(yt_df["created_at"]) 


In [18]:
sampled_video_ids = yt_df["video_id"].drop_duplicates().sample(10000, random_state=42)
yt_sample = yt_df[yt_df["video_id"].isin(sampled_video_ids)].copy()

print(f"Sampled videos   : {yt_sample['video_id'].nunique():,}")
print(f"Total rows       : {len(yt_sample):,}")
print(f"Memory usage     : {yt_sample.memory_usage(deep=True).sum()/1024/1024:.1f} MB")
print(f"Date range       : {yt_sample['created_at'].min()} → {yt_sample['created_at'].max()}")

Sampled videos   : 10,000
Total rows       : 1,685,568
Memory usage     : 141.5 MB
Date range       : 2021-01-01 00:00:28 → 2022-05-10 11:01:11


In [20]:
#Time series feature engineering:
def engineer_video_features(df):
    features = []
    
    for video_id, group in df.groupby("video_id"):
        group = group.sort_values("created_at")

        # Skip videos with less than 2 hours of data
        if len(group) < 2:
            continue
        
        # Hour of day when most activity happened
        peak_hour = group.loc[group["views"].diff().idxmax(), "created_at"].hour
        
        # How fast did likes grow vs views
        view_growth = group["views"].diff().fillna(0)
        like_growth = group["likes"].diff().fillna(0)
        
        # Max single hour spike in likes
        max_like_spike = like_growth.max()
        max_view_spike = view_growth.max()
        
        # Ratio of max spike to average growth
        avg_like_growth = like_growth.mean()
        avg_view_growth = view_growth.mean()
        
        features.append({
            "video_id"              : video_id,
            "total_views"           : group["views"].max(),
            "total_likes"           : group["likes"].max(),
            "total_comments"        : group["comments"].max(),
            "like_view_ratio"       : group["likes"].max() / (group["views"].max() + 1),
            "max_like_spike"        : max_like_spike,
            "max_view_spike"        : max_view_spike,
            "like_spike_ratio"      : max_like_spike / (avg_like_growth + 1),
            "view_spike_ratio"      : max_view_spike / (avg_view_growth + 1),
            "peak_hour"             : peak_hour,
            "hours_tracked"         : len(group),
        })
    
    return pd.DataFrame(features)

video_features = engineer_video_features(yt_sample)


In [21]:
video_features.head()

,video_id,total_views,total_likes,total_comments,like_view_ratio,max_like_spike,max_view_spike,like_spike_ratio,view_spike_ratio,peak_hour,hours_tracked
0,100012,3924,52.0,10.0,0.013248,13.0,414.0,10.184332,18.223718,18,170
1,100087,22384,147.0,257.0,0.006567,13.0,2110.0,7.222222,15.993401,0,170
2,100088,5198,89.0,46.0,0.017119,13.0,988.0,9.364407,32.772683,12,170
3,100106,6094,93.0,6.0,0.015258,18.0,970.0,12.540984,27.419355,19,170
4,100119,34569,304.0,176.0,0.008794,58.0,5882.0,23.309693,29.160421,17,170


In [31]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Select features for the model
feature_cols = [
    "like_view_ratio",
    "max_like_spike",
    "max_view_spike", 
    "like_spike_ratio",
    "view_spike_ratio",
    "peak_hour",
    "hours_tracked"
]

X = video_features[feature_cols].fillna(0)

# Scale features - important for Isolation Forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
# contamination = expected % of anomalies in data
# We estimate ~5% of videos have fake engagement
iso_forest = IsolationForest(contamination=0.08, random_state=42)
video_features["anomaly_score"] = iso_forest.fit_predict(X_scaled)
video_features["is_suspicious"] = video_features["anomaly_score"] == -1

print(f"Total videos analysed  : {len(video_features):,}")
print(f"Flagged as suspicious  : {video_features['is_suspicious'].sum():,}")
print(f"Suspicion rate         : {video_features['is_suspicious'].mean():.1%}")

Total videos analysed  : 9,999
Flagged as suspicious  : 800
Suspicion rate         : 8.0%


In [34]:
video_features.to_csv("../data/video_features_contamination_10.csv", index=False)

In [35]:
#Inspect flagged video
suspicious = video_features[video_features["is_suspicious"]==True].sort_values(
    "like_spike_ratio", ascending=False)
suspicious.to_csv("../data/suspicious_videos.csv", index=False)

normal = video_features[video_features["is_suspicious"]==False].sort_values(
    "like_spike_ratio", ascending=False)
normal.to_csv("../data/normal_videos.csv", index=False)

print("=== SUSPICIOUS VIDEOS (top 10 by like spike ratio) ===")
print(suspicious[["video_id","total_views","total_likes",
                   "like_view_ratio","like_spike_ratio",
                   "view_spike_ratio","peak_hour"]].head(10))


=== SUSPICIOUS VIDEOS (top 10 by like spike ratio) ===
      video_id  total_views  total_likes  like_view_ratio  like_spike_ratio  \
7037    293512        26804       8038.0         0.299869        111.758449   
9542    358378        10425        277.0         0.026568         97.360179   
1138    130877        18059       1305.0         0.072259         91.722099   
9611    360235         6577        245.0         0.037245         84.735202   
9631    360699         6988       1350.0         0.193161         79.400444   
1189    132271         9551        763.0         0.079879         79.345692   
9511    357691        10122       1131.0         0.111726         78.187614   
6147    269092        10353        564.0         0.054472         76.295181   
383     110994        30084        306.0         0.010171         73.010526   
1157    131486         3753        260.0         0.069259         71.767810   

      view_spike_ratio  peak_hour  
7037         14.813946         22  
954

In [36]:
def assign_tier(row):
    if not row["is_suspicious"]:
        return "clean"
    elif row["like_spike_ratio"] > 30:
        return "tier1_high_confidence"
    elif row["like_spike_ratio"] > 20:
        return "tier2_human_review"
    elif row["like_spike_ratio"] > 10:
        return "tier3_watchlist"
    else:
        return "tier4_ignore"

video_features["enforcement_tier"] = video_features.apply(assign_tier, axis=1)

print(video_features["enforcement_tier"].value_counts())

# Save everything
video_features.to_csv("../data/video_features_final.csv", index=False)
suspicious = video_features[video_features["is_suspicious"]==True].sort_values(
    "like_spike_ratio", ascending=False)
suspicious.to_csv("../data/suspicious_videos_tiered.csv", index=False)

print("\nFiles saved ✓")

enforcement_tier
clean                    9199
tier4_ignore              280
tier2_human_review        192
tier1_high_confidence     173
tier3_watchlist           155
Name: count, dtype: int64

Files saved ✓


In [26]:
print("\n=== NORMAL VIDEOS (sample) ===")
print(normal[["video_id","total_views","total_likes",
              "like_view_ratio","like_spike_ratio",
              "view_spike_ratio","peak_hour"]].head(10))


=== NORMAL VIDEOS (sample) ===
      video_id  total_views  total_likes  like_view_ratio  like_spike_ratio  \
7819    313661         6490        106.0         0.016330         69.360000   
267     107689        22861        140.0         0.006124         42.073579   
9618    360413         7294        190.0         0.026045         41.511628   
7221    297875         3093         58.0         0.018746         41.141553   
1172    131864         7729        709.0         0.091721         39.282869   
1624    144129        26826        191.0         0.007120         38.323699   
7857    314784        10655        297.0         0.027872         37.470998   
9617    360388        19841        525.0         0.026459         37.110187   
1770    148174         5378        130.0         0.024168         35.924528   
7041    293675        10885        241.0         0.022139         35.721519   

      view_spike_ratio  peak_hour  
7819         15.025898          9  
267          19.487314    

In [27]:
print(f"\nAvg like_spike_ratio - Suspicious : {suspicious['like_spike_ratio'].mean():.1f}")
print(f"Avg like_spike_ratio - Normal     : {normal['like_spike_ratio'].mean():.1f}")


Avg like_spike_ratio - Suspicious : 22.4
Avg like_spike_ratio - Normal     : 4.1


In [28]:
def add_sustainability(df):
    results = []
    
    for video_id, group in df.groupby("video_id"):
        group = group.sort_values("created_at")
        
        if len(group) < 3:
            continue
        
        like_growth = group["likes"].diff().fillna(0)
        
        # Find the hour of maximum spike
        spike_idx = like_growth.idxmax()
        spike_pos = group.index.get_loc(spike_idx)
        
        # Look at what happens AFTER the spike
        # If bot: likes drop back to near zero immediately
        # If viral: likes stay elevated for several hours
        if spike_pos + 3 < len(group):
            after_spike = like_growth.iloc[spike_pos+1 : spike_pos+4].mean()
            spike_value = like_growth.iloc[spike_pos]
            
            # Ratio of post-spike activity to spike itself
            # Low ratio = dropped immediately = bot signature
            sustainability = after_spike / (spike_value + 1)
        else:
            sustainability = 0
            
        results.append({
            "video_id"       : video_id,
            "sustainability" : sustainability
        })
    
    return pd.DataFrame(results)

print("Calculating sustainability... takes about a minute")
sustainability_df = add_sustainability(yt_sample)
video_features = video_features.merge(sustainability_df, on="video_id", how="left")

print(f"Feature added ✓")
print(f"\nSustainability stats:")
print(video_features["sustainability"].describe())

Calculating sustainability... takes about a minute
Feature added ✓

Sustainability stats:
count    9999.000000
mean        0.134525
std         0.216955
min        -0.619048
25%         0.000000
50%         0.000000
75%         0.238492
max         0.952381
Name: sustainability, dtype: float64


In [29]:
print(f"\nSuspicious avg sustainability: {video_features[video_features['is_suspicious']]['sustainability'].mean():.3f}")
print(f"Normal avg sustainability    : {video_features[~video_features['is_suspicious']]['sustainability'].mean():.3f}")


Suspicious avg sustainability: 0.335
Normal avg sustainability    : 0.124


In [30]:
feature_cols = [
    "like_view_ratio",
    "max_like_spike",
    "max_view_spike",
    "like_spike_ratio",
    "view_spike_ratio",
    "peak_hour",
    "hours_tracked",
    "sustainability"
]

X = video_features[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest2 = IsolationForest(contamination=0.05, random_state=42)
video_features["anomaly_score_v2"] = iso_forest2.fit_predict(X_scaled)
video_features["is_suspicious_v2"] = video_features["anomaly_score_v2"] == -1

# Compare v1 vs v2 flagging
both_flagged    = ((video_features["is_suspicious"]) & (video_features["is_suspicious_v2"])).sum()
only_v1_flagged = ((video_features["is_suspicious"]) & (~video_features["is_suspicious_v2"])).sum()
only_v2_flagged = ((~video_features["is_suspicious"]) & (video_features["is_suspicious_v2"])).sum()

print(f"Flagged by both models    : {both_flagged}")
print(f"Dropped by v2 (v1 only)  : {only_v1_flagged}")
print(f"New flags in v2           : {only_v2_flagged}")

Flagged by both models    : 424
Dropped by v2 (v1 only)  : 76
New flags in v2           : 76
